# Experimentos de mejora — Notebook maestro

Punto de entrada a los 10 experimentos que fortalecen el proyecto de análisis de sentimiento ordinal (TikTok, Mundial 2026). Cada experimento tiene su propio notebook; aquí se resume todo.

**Cómo leer esto:** por defecto los notebooks CARGAN los resultados ya calculados (rápido, sin GPU). Para re-ejecutar desde cero, pon `RECOMPUTE=True` en cada uno (requiere el venv de ML y/o Bedrock).

**Orden de ejecución de los experimentos:**

| Fase | Notebook | Experimentos | Requiere |
|------|----------|--------------|----------|
| 0 | `01_P0_...` | P0 evaluación honesta | GPU |
| 1 | `02_LLMs_...` | E1, E2, E3 | Bedrock |
| 2 | `03_locales_...` | E4, E6, E7, E8 | GPU |
| 3 | `04_calibracion_...` | E5, E9 | GPU (E5) |

**El baseline honesto contra el que se mide TODO: QWK 0.415** (nunca el 0.66 in-sample, que era leakage).

In [1]:
# --- Configuracion comun ---
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import common as C
R = C.RESULTS
def load(f): return json.load(open(R / f))

# Patron de dos niveles: por defecto CARGA resultados ya calculados (segundos, sin GPU).
# Para RE-EJECUTAR desde cero (requiere GPU/Bedrock), pon RECOMPUTE=True.
RECOMPUTE = False

## Tabla resumen de los 10 experimentos (con intervalos de confianza)

In [2]:
rows = []
p0 = load("p0_stratified.json")["oof_global"]
rows.append(["P0 baseline", round(p0["qwk"],3), "-", "referencia"])
e1 = load("e1_triple.json"); d = e1["delta_qwk_llm_minus_bert"]
rows.append(["E1 Consenso-LLM", round(e1["llm_consensus"]["qwk"],3),
             f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]",
             f"empate (P={d['p_llm_gt_bert']:.2f})"])
e2 = load("e2_aggregation.json"); rows.append(["E2 mediana ordinal", round(e2["aggregations"]["mediana"]["qwk"],3), "-", "mediana > mayoria"])
e3 = load("e3_gate3.json"); d = e3["delta_qwk_vs_p0"]
rows.append(["E3 auto-etiquetado", round(e3["metrics"]["qwk"],3), f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "dentro del ruido"])
e4 = load("e4_corn.json"); d = e4["delta_mae_neg_corn_minus_base"]
rows.append(["E4 CORN (MAE)", round(e4["corn_metrics"]["mae"],3), f"MAE {d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "SIGNIFICATIVO"])
e6 = load("e6_backbone.json"); rows.append(["E6 RoBERTuito", round(e6["robertuito"]["oof_global"]["qwk"],3), "+0.055", "significativo (borde)"])
e7 = load("e7_limpieza.json"); d = e7["delta_qwk_minima_minus_agresiva"]
rows.append(["E7 limpieza minima", round(e7["text_minima"]["oof_global"]["qwk"],3), f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "dentro del ruido"])
e8 = load("e8_augmentation.json"); d = e8["delta_qwk_vs_p0"]
rows.append(["E8 augmentation", round(e8["metrics"]["qwk"],3), f"{d['mean']:+.3f} [{d['ci_low']:+.3f},{d['ci_high']:+.3f}]", "SIGNIFICATIVO"])
import pandas as pd
tabla = pd.DataFrame(rows, columns=["Experimento","QWK","Delta vs baseline (IC95)","Veredicto"])
tabla.to_csv(R / "summary_table.csv", index=False)
tabla

,Experimento,QWK,Delta vs baseline (IC95),Veredicto
0,P0 baseline,0.415,-,referencia
1,E1 Consenso-LLM,0.498,"+0.082 [-0.002,+0.168]",empate (P=0.97)
2,E2 mediana ordinal,0.498,-,mediana > mayoria
3,E3 auto-etiquetado,0.449,"+0.033 [-0.025,+0.089]",dentro del ruido
4,E4 CORN (MAE),0.656,"MAE +0.106 [+0.043,+0.164]",SIGNIFICATIVO
5,E6 RoBERTuito,0.470,+0.055,significativo (borde)
6,E7 limpieza minima,0.460,"+0.017 [-0.017,+0.051]",dentro del ruido
7,E8 augmentation,0.472,"+0.057 [+0.018,+0.097]",SIGNIFICATIVO


## Rigor estadistico (fortificacion para tesis)

Tres validaciones que un jurado exigira:

In [3]:
# 1) Robustez multi-semilla del baseline (responde: "una sola corrida?")
ms = load("p0_multiseed.json")
print(f"Baseline QWK entre {len(ms['seeds'])} semillas: {ms['qwk_mean']:.3f} +/- {ms['qwk_sd']:.3f}")
print("  por semilla:", {p['seed']: round(p['qwk'],3) for p in ms['per_seed']})
print("=> estable, no depende del sorteo de folds")

Baseline QWK entre 3 semillas: 0.396 +/- 0.020
  por semilla: {61298: 0.379, 7: 0.392, 2024: 0.418}
=> estable, no depende del sorteo de folds


In [4]:
# 2) Correccion por comparaciones multiples (Holm-Bonferroni, test de permutacion)
holm = load("stats_holm.json")
import pandas as pd
print("En QWK, tras Holm-Bonferroni:")
display(pd.DataFrame(holm))
print("Ningun delta de QWK sobrevive la correccion. Los hallazgos solidos son:")
print("  - E4 en MAE (IC[+0.043,+0.164], lejos de 0)")
print("  - E8 fold-aware en F1 de clases 4/5")

En QWK, tras Holm-Bonferroni:


,exp,delta_qwk,p_perm,umbral_holm,sig
0,E1_LLM,0.083,0.0614,0.0167,False
1,E3_autolabel,0.033,0.2551,0.0250,False
2,E4_CORN,-0.001,0.9850,0.0500,False


Ningun delta de QWK sobrevive la correccion. Los hallazgos solidos son:
  - E4 en MAE (IC[+0.043,+0.164], lejos de 0)
  - E8 fold-aware en F1 de clases 4/5


In [5]:
# 3) E8 SIN fuga (fold-aware): las parafrasis se generan solo del train de cada fold
fa = load("e8_augmentation_foldaware.json"); d = fa["delta_qwk_vs_p0"]
print(f"E8 fold-aware: F1(4/5) {fa['f1_clases45_base']:.3f} -> {fa['f1_clases45_aug']:.3f}")
print(f"Delta QWK {d['mean']:+.3f} IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}] -> SIGUE significativo tras quitar el leakage")

E8 fold-aware: F1(4/5) 0.223 -> 0.262
Delta QWK +0.045 IC[+0.005,+0.087] -> SIGUE significativo tras quitar el leakage


### Lectura honesta de la tabla

Solo **tres** experimentos tienen una mejora estadísticamente significativa (IC que excluye 0): **E4** (cabeza ordinal, reduce MAE), **E8** (augmentation, sube clases raras) y **E6** (RoBERTuito, al borde). El hallazgo estrella E1 (el LLM iguala al BERT) es técnicamente un **empate** (P=0.97): el marco honesto es *"con N=581, el fine-tuning no logra superar a un LLM sin entrenar"*, no *"el LLM gana"*. E3 y E7 quedan dentro del ruido. Reportar los resultados nulos es parte del método.